# Mistral OCR AI Gateway diagnostics

Run this notebook one cell at a time. Each checkpoint isolates one layer of the request path:

1. Local configuration
2. DNS and TLS connectivity
3. Gateway behavior without a subscription key
4. APIM subscription authorization
5. APIM-to-Foundry authentication
6. Optional direct Foundry comparison
7. A real OCR request

The notebook never displays access tokens or subscription keys. Stop at the first failed checkpoint and use its interpretation before continuing.

## 0. Setup

Copy `.env.example` to `.env` and set the real values only in `.env`. For an integrated Foundry AI Gateway, start with `AZURE_AI_GATEWAY_AUTH_MODE=gateway_managed_identity`. Use `passthrough_entra` only when APIM is intentionally configured to forward the caller's bearer token to Foundry.

In [1]:
import base64
import json
import mimetypes
import os
import socket
import ssl
from pathlib import Path
from typing import Any
from urllib.parse import urlparse

import requests
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()

True

## 1. Validate local configuration

This verifies that the gateway endpoint includes an API path suffix and that required values are present. Secret values are not printed.

In [2]:
GATEWAY_ENDPOINT = os.getenv("AZURE_AI_GATEWAY_ENDPOINT", "").rstrip("/")
GATEWAY_SUBSCRIPTION_KEY = os.getenv("AZURE_AI_GATEWAY_SUBSCRIPTION_KEY", "")
GATEWAY_KEY_HEADER = os.getenv("AZURE_AI_GATEWAY_KEY_HEADER", "api-key")
FOUNDRY_ENDPOINT = os.getenv("AZURE_FOUNDRY_ENDPOINT", "").rstrip("/")
OCR_DEPLOYMENT = os.getenv("AZURE_MISTRAL_OCR_DEPLOYMENT", "")
AUTH_MODE = os.getenv("AZURE_AI_GATEWAY_AUTH_MODE", "gateway_managed_identity")
OCR_PATH = "/providers/mistral/azure/ocr"
GATEWAY_OCR_ENDPOINT = f"{GATEWAY_ENDPOINT}{OCR_PATH}"

required = {
    "AZURE_AI_GATEWAY_ENDPOINT": GATEWAY_ENDPOINT,
    "AZURE_AI_GATEWAY_SUBSCRIPTION_KEY": GATEWAY_SUBSCRIPTION_KEY,
    "AZURE_MISTRAL_OCR_DEPLOYMENT": OCR_DEPLOYMENT,
}
missing = [name for name, value in required.items() if not value]
if missing:
    raise RuntimeError(f"Missing .env values: {', '.join(missing)}")

gateway_url = urlparse(GATEWAY_ENDPOINT)
if gateway_url.scheme != "https" or not gateway_url.hostname:
    raise ValueError("AZURE_AI_GATEWAY_ENDPOINT must be an absolute HTTPS URL.")
if not gateway_url.path.strip("/"):
    raise ValueError("AZURE_AI_GATEWAY_ENDPOINT must include the generated APIM API path suffix.")
if GATEWAY_ENDPOINT.endswith(OCR_PATH):
    raise ValueError(f"Remove {OCR_PATH} from AZURE_AI_GATEWAY_ENDPOINT; the notebook appends it.")
if AUTH_MODE not in {"gateway_managed_identity", "passthrough_entra"}:
    raise ValueError("AZURE_AI_GATEWAY_AUTH_MODE must be gateway_managed_identity or passthrough_entra.")

print("PASS: Required gateway configuration is present.")
print(f"Gateway host: {gateway_url.hostname}")
print(f"Gateway API path: {gateway_url.path}")
print(f"OCR operation path: {OCR_PATH}")
print(f"Backend auth mode: {AUTH_MODE}")
print(f"Gateway key header: {GATEWAY_KEY_HEADER}")
print(f"Subscription key loaded: yes ({len(GATEWAY_SUBSCRIPTION_KEY)} characters; value hidden)")
print(f"Optional direct Foundry comparison configured: {'yes' if FOUNDRY_ENDPOINT else 'no'}")

PASS: Required gateway configuration is present.
Gateway host: aigw-ckriutz.azure-api.net
Gateway API path: /mistral-ocr
OCR operation path: /providers/mistral/azure/ocr
Backend auth mode: gateway_managed_identity
Gateway key header: Ocp-Apim-Subscription-Key
Subscription key loaded: yes (32 characters; value hidden)
Optional direct Foundry comparison configured: no


## 2. Test DNS and TLS connectivity

This does not call the API. It confirms that the gateway hostname resolves and that the machine can establish TLS. A failure here points to DNS, firewall, proxy, or private-network configuration rather than an API key or Foundry permission.

In [3]:
gateway_host = gateway_url.hostname
addresses = sorted({item[4][0] for item in socket.getaddrinfo(gateway_host, 443, type=socket.SOCK_STREAM)})
print(f"PASS: DNS resolved {gateway_host} to {', '.join(addresses)}")

tls_context = ssl.create_default_context()
with socket.create_connection((gateway_host, 443), timeout=10) as tcp_socket:
    with tls_context.wrap_socket(tcp_socket, server_hostname=gateway_host) as tls_socket:
        print(f"PASS: TLS connected using {tls_socket.version()} and {tls_socket.cipher()[0]}")

PASS: DNS resolved aigw-ckriutz.azure-api.net to 74.157.155.66
PASS: TLS connected using TLSv1.3 and TLS_AES_256_GCM_SHA384


## Response diagnostic helper

The following probes intentionally send an empty JSON body. A `400` or `422` can be a successful diagnostic result: it means the request passed the tested authentication layer and reached API payload validation.

In [4]:
def classify_response(response: requests.Response) -> str:
    body = response.text.lower()
    if response.status_code in {400, 422}:
        return "PASS: The request reached API/backend payload validation."
    if response.status_code < 400:
        return "PASS: The request was accepted."
    if response.status_code == 401 and "access denied" in body:
        return "FAIL at APIM client authentication: check subscription key scope, state, and header."
    if response.status_code == 401:
        return "FAIL at authentication: inspect WWW-Authenticate and APIM trace to distinguish gateway from backend."
    if response.status_code == 403:
        return "FAIL at authorization: check the APIM subscription or backend managed-identity RBAC."
    if response.status_code == 404:
        return "FAIL at routing: check the generated API suffix and OCR operation path."
    if response.status_code == 405:
        return "FAIL at operation configuration: POST is not registered for this APIM route."
    if response.status_code == 429:
        return "FAIL at quota/rate limiting: inspect APIM policies and Foundry deployment quota."
    if response.status_code >= 500:
        return "FAIL at gateway/backend execution: inspect APIM gateway logs and backend connectivity."
    return "FAIL: Inspect the bounded response body and APIM trace."


def show_response(label: str, response: requests.Response) -> None:
    print(f"{label}: HTTP {response.status_code}")
    print(classify_response(response))
    print(f"APIM request ID: {response.headers.get('apim-request-id', 'not returned')}")
    print(f"Backend request ID: {response.headers.get('x-ms-request-id', 'not returned')}")
    print(f"WWW-Authenticate: {response.headers.get('www-authenticate', 'not returned')}")
    print("Response body (bounded):")
    print(response.text[:1500] or "<empty>")


def probe(headers: dict[str, str] | None = None) -> requests.Response:
    return requests.post(
        GATEWAY_OCR_ENDPOINT,
        headers={"Content-Type": "application/json", **(headers or {})},
        json={},
        timeout=30,
    )

## 3. Probe APIM without a subscription key

If **Subscription required** is enabled, the expected result is a gateway-level `401 Access Denied`. That expected failure proves DNS, TLS, gateway hostname, and at least part of the API routing are reachable.

In [5]:
no_key_response = probe()
show_response("No-key probe", no_key_response)

if no_key_response.status_code == 401 and "access denied" in no_key_response.text.lower():
    print("CHECKPOINT PASS: The expected no-key rejection confirms that this API requires a gateway key.")
    challenge = no_key_response.headers.get("www-authenticate", "")
    expected_fragment = f'name="{GATEWAY_KEY_HEADER}"'
    if challenge and expected_fragment not in challenge:
        print(f"WARNING: The gateway challenge does not request the configured {GATEWAY_KEY_HEADER!r} header.")
        print("Set AZURE_AI_GATEWAY_KEY_HEADER to the name shown in WWW-Authenticate.")
else:
    print("NOTE: The API did not return the usual missing-subscription-key response; continue to compare results.")

No-key probe: HTTP 401
FAIL at APIM client authentication: check subscription key scope, state, and header.
APIM request ID: not returned
Backend request ID: not returned
WWW-Authenticate: AzureApiManagementKey realm="https://aigw-ckriutz.azure-api.net/mistral-ocr",name="Ocp-Apim-Subscription-Key",type="header"
Response body (bounded):
{ "statusCode": 401, "message": "Access denied due to missing subscription key. Make sure to include subscription key when making requests to an API." }
CHECKPOINT PASS: The expected no-key rejection confirms that this API requires a gateway key.


## 4. Probe APIM with only the subscription key

This isolates client-to-gateway authorization. The response should change from APIM `401 Access Denied` to a backend/authentication response or payload-validation error. If it stays `401 Access Denied`, verify that the subscription is active and scoped to this API or its product.

In [6]:
subscription_headers = {GATEWAY_KEY_HEADER: GATEWAY_SUBSCRIPTION_KEY}
key_response = probe(subscription_headers)
show_response("Subscription-key probe", key_response)

if key_response.status_code == no_key_response.status_code and key_response.text == no_key_response.text:
    print("FAIL: Adding the key made no observable difference. Check the key, subscription scope, and header name.")
else:
    print("PASS: Adding the key changed the gateway response.")

Subscription-key probe: HTTP 400
PASS: The request reached API/backend payload validation.
APIM request ID: 1bf9587b-10e1-4125-b3be-8f19a5b75fea
Backend request ID: not returned
WWW-Authenticate: not returned
Response body (bounded):
{"error":{"code":"no_model_name","message":"Could not find a reference to a model in the request. Please provide a model name as a string under the 'model' parameter in the request body.","details":"Could not find a reference to a model in the request. Please provide a model name as a string under the 'model' parameter in the request body."}}
PASS: Adding the key changed the gateway response.


## 5. Test the configured APIM-to-Foundry authentication mode

- `gateway_managed_identity`: the key-only result above is the complete gateway request. A backend `401` or `403` means the generated APIM backend identity/policy or RBAC needs attention.
- `passthrough_entra`: this checkpoint adds a local Entra token for Foundry and compares the result.

In [7]:
TOKEN_SCOPE = "https://ai.azure.com/.default"
credential: DefaultAzureCredential | None = None


def get_credential() -> DefaultAzureCredential:
    global credential
    if credential is None:
        credential = DefaultAzureCredential()
    return credential


def configured_gateway_headers() -> dict[str, str]:
    headers = dict(subscription_headers)
    if AUTH_MODE == "passthrough_entra":
        token = get_credential().get_token(TOKEN_SCOPE)
        headers["Authorization"] = f"Bearer {token.token}"
    return headers


if AUTH_MODE == "passthrough_entra":
    print("PASS: Acquired a local Entra token (value hidden).")
    backend_auth_response = probe(configured_gateway_headers())
    show_response("Subscription-key plus Entra probe", backend_auth_response)
else:
    backend_auth_response = key_response
    print("Gateway managed identity selected; the key-only probe is the applicable backend-auth test.")
    print(classify_response(backend_auth_response))

Gateway managed identity selected; the key-only probe is the applicable backend-auth test.
PASS: The request reached API/backend payload validation.


## 6. Optional direct Foundry comparison

If `AZURE_FOUNDRY_ENDPOINT` is set, this sends the same empty-payload probe directly to Foundry with your local Entra identity. A direct `400`/`422` combined with a gateway backend `401`/`403` isolates the problem to APIM backend authentication or RBAC. This is a diagnostic comparison, not the intended production route.

In [ ]:
if not FOUNDRY_ENDPOINT:
    print("SKIPPED: Set AZURE_FOUNDRY_ENDPOINT to run the direct Foundry comparison.")
else:
    foundry_url = urlparse(FOUNDRY_ENDPOINT)
    if foundry_url.scheme != "https" or not foundry_url.hostname:
        raise ValueError("AZURE_FOUNDRY_ENDPOINT must be an absolute HTTPS URL.")

    direct_token = get_credential().get_token(TOKEN_SCOPE)
    direct_response = requests.post(
        f"{FOUNDRY_ENDPOINT}{OCR_PATH}",
        headers={
            "Authorization": f"Bearer {direct_token.token}",
            "Content-Type": "application/json",
        },
        json={},
        timeout=30,
    )
    show_response("Direct Foundry probe", direct_response)

## 7. Run one real OCR request

Run this only after the applicable malformed-payload probe reaches `400`/`422` or otherwise shows that gateway and backend authentication succeeded.

In [8]:
DOCUMENT_PATH = Path("documents/AdventureWorksPO_HROrder.png")


def document_data_url(path: Path) -> str:
    if not path.is_file():
        raise FileNotFoundError(f"Document not found: {path.resolve()}")
    media_type, _ = mimetypes.guess_type(path.name)
    if media_type is None:
        raise ValueError(f"Could not determine media type for {path.name}")
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:{media_type};base64,{encoded}"


payload: dict[str, Any] = {
    "model": OCR_DEPLOYMENT,
    "document": {
        "type": "document_url",
        "document_url": document_data_url(DOCUMENT_PATH),
    },
    "include_image_base64": False,
}

ocr_response = requests.post(
    GATEWAY_OCR_ENDPOINT,
    headers={"Content-Type": "application/json", **configured_gateway_headers()},
    json=payload,
    timeout=180,
)
show_response("Real OCR request", ocr_response)
ocr_response.raise_for_status()
ocr_result = ocr_response.json()

pages = ocr_result.get("pages", [])
display({
    "page_count": len(pages),
    "markdown_characters": sum(len(page.get("markdown", "")) for page in pages),
})
if pages:
    display(Markdown(pages[0].get("markdown", "")))

Real OCR request: HTTP 200
PASS: The request was accepted.
APIM request ID: b9a7847a-7df9-438a-b560-5a18908a663c
Backend request ID: not returned
WWW-Authenticate: not returned
Response body (bounded):
{"pages":[{"index":0,"markdown":"ADVENTURE WORKS\n\nAdventure Works\n\n4821 Innovation Parkway, Suite 300,\n\nCharlotte, NC 28273\n\nUSA\n\n|  SUPPLIER | Purchase Order Details  |\n| --- | --- |\n|  Stuff We All Get Inc. 1313 N. South River Rd. Gatsby, ID 96302 | PO number: swag-PO-1199 Created By: Mindy Smith Dept: HR  |\n\n|  NOTES  |\n| --- |\n|  |\n\n|  Item Code | Description | Quantity | Unit Price | Line Total  |\n| --- | --- | --- | --- | --- |\n|  SG-3003 | Shirt Shaped Shirts | 30 | 17.99 | 539.70  |\n|  SG-2119 | Box of Cute Cubes | 100 | 1.20 | 120.00  |\n|  SG-0401 | Branded Playing Cards | 75 | 3.50 | 262.50  |\n|  |   |   |   |   |\n|  SUBTOTAL |   |   |   | 922.20  |\n|  TAX (7%) |   |   |   | 64.55  |\n|  GRAND TOTAL |   |   |   | 986.75  |","images":[],"tables":[],"hype

{'page_count': 1, 'markdown_characters': 708}

ADVENTURE WORKS

Adventure Works

4821 Innovation Parkway, Suite 300,

Charlotte, NC 28273

USA

|  SUPPLIER | Purchase Order Details  |
| --- | --- |
|  Stuff We All Get Inc. 1313 N. South River Rd. Gatsby, ID 96302 | PO number: swag-PO-1199 Created By: Mindy Smith Dept: HR  |

|  NOTES  |
| --- |
|  |

|  Item Code | Description | Quantity | Unit Price | Line Total  |
| --- | --- | --- | --- | --- |
|  SG-3003 | Shirt Shaped Shirts | 30 | 17.99 | 539.70  |
|  SG-2119 | Box of Cute Cubes | 100 | 1.20 | 120.00  |
|  SG-0401 | Branded Playing Cards | 75 | 3.50 | 262.50  |
|  |   |   |   |   |
|  SUBTOTAL |   |   |   | 922.20  |
|  TAX (7%) |   |   |   | 64.55  |
|  GRAND TOTAL |   |   |   | 986.75  |

## Decision table

| First failing checkpoint | Configuration area to fix |
|---|---|
| Local configuration | `.env` names, gateway API suffix, auth mode |
| DNS/TLS | DNS, firewall, proxy, private endpoint, APIM network access |
| No-key probe cannot reach APIM | Gateway URL or generated API route |
| Key probe remains `401 Access Denied` | APIM subscription state, key, API/product scope |
| Key accepted, then backend `401` | Generated backend authentication/policy |
| Key accepted, then backend `403` | APIM managed-identity Foundry RBAC |
| Direct Foundry fails too | Foundry endpoint, deployment access, local identity/RBAC |
| Malformed probe works, real OCR fails | Deployment name, payload, document format/size, model quota |

For the failing request, copy the displayed APIM request ID and inspect **API Management → Monitoring → Logs** or enable an APIM trace. Do not share subscription keys, bearer tokens, or complete request headers.